# Step 4 — View 묶음 · Point 충돌 · 선택 (노트북)

API(`POST /api/v1/rag/v4/views` · `/views/select` · `/views/points`)·CLI(`python -m backend.rag.view "질문"`)와 같은 함수.

| 단계 | 함수 | 바꿔 볼 설정 (`configs/query_v4.yaml`, `overrides`) |
|---|---|---|
| 2-2 View 묶음 | `build_viewset(질문 · analyze 결과 · view_bundle)` | `view_n_views`, `view_top_k`, `view_retrieval_config` |
| 2-3 Point | `record_points(record, filters)` | — |
| 2-3 충돌 | `detect` (build_viewset 안) · `view_usage` | `conflict_rel_tol`, `conflict_abs_tol`, `conflict_allow_partial` |
| 선택 | `on_ambiguous=콜백` · `apply_selections` · `resolve(token, option_id)` | `selection_ttl_hours` |

> 저장소 루트에서. 데이터·HCX 없이 해 보려면 `V4_NB_FAKE=1` — 가짜 데이터에 **같은 교차점 값이 다른 표**(DT_Y001)와
> **같은 값을 천명 단위로 둔 표**(DT_Y002)를 넣어 충돌을 재현한다.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parents[1]
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
USE_FAKE = os.getenv("V4_NB_FAKE", "0") == "1"
if USE_FAKE:
    import tempfile
    from tests import fake_kosis
    from tests.v4.test_view_v4 import build_conflict_data
    tmp = Path(tempfile.mkdtemp()) / "storage"
    build_conflict_data(tmp / "kosis_data")
    for k in ("KOSIS_DATA_DIR", "KOSIS_DATA", "VIEW_OUT_DIR", "TABLE_INDEX_DIR", "VIEW_EMBED_CACHE", "CSV_INDEX_DIR", "RUNS_DIR"):
        os.environ[k] = ""
    os.environ["STORAGE_DIR"], os.environ["HCX_API_KEY"] = str(tmp), "test"
    import backend.core.llm as llm, backend.rag.retrieval.dense as dense
    llm._post, dense._post = fake_kosis.fake_hcx_post, fake_kosis.fake_dense_post
    from backend.rag.ingestion import meta_profile, table_text
    meta_profile.build_all(); table_text.build_index(progress=lambda m: None)
    from backend.rag.csv_index import build_index
    build_index(scope="all", log=lambda m: None)
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from backend.rag.view import build_viewset, apply_selections, record_points, resolve
from backend.core.tracking import Tracer

## 1. View 묶음 + 충돌 판정 — view_bundle 로 (질문 문자열·analyze 결과도 됨)

In [ ]:
VB = {"query_id": "nb1", "original": "2025년 대전 인구 수", "search_text": "2025년 대전광역시 인구수",
      "filters": {"시점": [2025], "지역": ["대전광역시"], "지표": ["인구수"], "의도": "값 조회"}}
tr = Tracer("view_nb")
vs = build_viewset(VB, n_views=5, tracer=tr)
u = vs["units"][0]; cf = u["conflict"]
print("상태:", vs["status"], "| 단위:", u["unit_id"], cf["status"], "|", cf["notes"])
pd.DataFrame(cf["views"])[["tbl_id", "rank", "usable", "used", "n_points", "notes"]]

필수 메타를 다 쓴 View(`usable`)만 비교한다. 같은 교차점에서 값이 다른 곳:

In [ ]:
pd.DataFrame([{"교차점": d["key_str"], **{x["tbl_id"]: f"{x['value']} {x['unit']}" for x in d["values"]}}
              for d in cf["differing_points"]])

## 2. 선택지 — 값이 같은 표끼리 한 묶음, 주석·표 설명·산출 기준과 함께

In [ ]:
for o in cf["options"]:
    print(f"{o['option_id']}  {o['tbl_nm']}  {o['tbl_ids']}")
    for fn in o["footnotes"]:
        print(f"    ※ [{fn['type']} {fn['target']}] {fn['text']}")
    print("    기준:", {k: v for k, v in o["basis"].items() if v})
print("\n설명(opt-1):\n" + cf["options"][0]["description"])

## 3. 선택 — 세 가지 방법

1. 콜백: `build_viewset(…, on_ambiguous=함수)` — 함수가 충돌 dict 를 받아 option_id 를 돌려줌 (노트북 위젯·input() 등)
2. 나중에: `apply_selections(vs, [{selection_token, option_id}])`
3. 토큰만: `resolve(token, option_id)` — 다른 실행에서 받은 토큰이어도 좌표로 View 를 다시 만든다

In [ ]:
def pick_by_unit(conflict):            # 예: 단위가 '천명' 이 아닌 선택지 중 검색 순위가 가장 높은 것
    for o in conflict["options"]:
        if o["basis"].get("단위") != "천명":
            return o["option_id"]
    return conflict["options"][0]["option_id"]

vs_cb = build_viewset(VB, n_views=5, on_ambiguous=pick_by_unit)
print("콜백:", vs_cb["status"], vs_cb["units"][0]["conflict"]["selected_option"], [r["source"]["tbl_id"] for r in vs_cb["records"]])

vs2, applied = apply_selections(vs, [{"selection_token": cf["selection_token"], "option_id": "opt-2"}])
print("나중에:", vs2["status"], [r["source"]["tbl_id"] for r in vs2["records"]], "| 뺀 것:", applied[0]["dropped_ids"])

r = resolve(cf["selection_token"], "opt-1")
print("토큰만:", [x["id"] for x in r["records"]], "| 다시 만듦:", r["rebuilt"])

## 4. Point 직접 보기 · 허용 오차 바꾸기

In [ ]:
rec = u["views"][0]["record"]
pts, notes = record_points(rec, VB["filters"])
pd.DataFrame([p.as_dict() for p in pts])[["key_str", "raw_value", "unit", "value", "labels"]]

In [ ]:
loose = build_viewset(VB, n_views=5, overrides={"conflict_rel_tol": 5.0})     # 500% 까지 같은 값으로 봄 (실험용)
print(loose["status"], loose["units"][0]["conflict"]["notes"])
pd.DataFrame(tr.as_list())[["step", "params", "output", "seconds"]]